# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and process the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library. The dataset was collected via survey and includes demographic characteristics and responses to psychological assessments including GAD-7, PHQ-9, and PSQ.

### Dataset Source

- Croissant schema: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)
- License: [Open Data Commons Attribution License](https://opendatacommons.org/licenses/by/1-0/)

We follow step-by-step: loading, record set and field inspection, DataFrame extraction, exploratory data analysis, and visualization.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant pandas --quiet

## 1. Data Loading
Load the dataset metadata and inspect its description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata object
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Let's enumerate available record sets, their fields, and reference their `@id`s (as required for clean and reproducible extraction and processing).

In [ ]:
from pprint import pprint

# List all record sets with their @id and fields
record_sets = [rs for rs in metadata.record_sets]
print(f"Found {len(record_sets)} record set(s).\n")
for i, rs in enumerate(record_sets):
    print(f"Record set {i+1}:")
    print(f"  name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Number of fields: {len(rs.fields)}")
    for field in rs.fields:
        print(f"    - field name: {field.name}, @id: {field.id}, dataType: {field.data_type}")
    print()

## 3. Data Extraction
Let's extract the full main record set (survey responses).

> **Note:** Use the primary survey record set's `@id` found above. All field and column references use their `@id`s. For analysis, you can extract all record sets into DataFrames, but typically the main survey table is the focus.


In [ ]:
# Choose main record set, e.g., the main survey table.
# If only one record set, use that one; else pick by name or column list
rs_id = None
for rs in metadata.record_sets:
    # Find the record set that most likely contains survey data
    if 'survey' in rs.name.lower() or 'responses' in rs.name.lower() or len(metadata.record_sets) == 1:
        rs_id = rs.id
        break
if rs_id is None:
    rs_id = metadata.record_sets[0].id
print(f"Using record set: {rs_id}")

records = list(dataset.records(record_set=rs_id))
df = pd.DataFrame(records)

print(f"Loaded {len(df)} records.")
print("Available columns / field @id:")
pprint(list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)
Typical EDA operations:
- Filter for respondents above/below a threshold (e.g., GAD-7 sum or PHQ-9 sum).
- Normalize a numeric column (e.g., PHQ-9/GAD-7/PSQ score).
- Group by a demographic field (e.g., age, level_of_education, or gender).

**All fields referenced by `@id`.**

In [ ]:
# For demonstration, try GAD-7 SCORE and Gender
# Identify the field @ids from the column list displayed above (actual @ids in your export)

# Example guess: find a numeric field for GAD-7 total and group by gender or education
numeric_field_id_candidates = [col for col in df.columns if 'gad' in col.lower() and ('score' in col.lower() or 'sum' in col.lower())]
if not numeric_field_id_candidates:
    numeric_field_id_candidates = [col for col in df.columns if col.lower().startswith('gad') or col.lower().startswith('phq')]
numeric_field_id = numeric_field_id_candidates[0] # Pick the first likely candidate

print(f"Using numeric_field_id: {numeric_field_id}")

group_field_id_candidates = [col for col in df.columns if 'gender' in col.lower() or 'education' in col.lower() or 'age' in col.lower()]
if group_field_id_candidates:
    group_field_id = group_field_id_candidates[0]
    print(f"Using group_field_id: {group_field_id}")
else:
    group_field_id = df.columns[0]  # Fallback
    print(f"[Fallback] Using {group_field_id}")

# Filter by a threshold on the numeric field (e.g., GAD-7 sum > 10: clinically significant anxiety)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()

print(f"Number of records with {numeric_field_id} > {threshold}: {len(filtered_df)}")

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records (first 5 rows):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count'])
    print(f"Grouped by {group_field_id} (mean and count):\n")
    print(grouped_df.head())

## 5. Visualization
Let's plot the distribution of the selected score, and the mean score by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of scores
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Number of participants")
plt.show()

# Boxplot by group
if group_field_id in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated:
- How to use `mlcroissant` to reproducibly load and inspect a real-world public health survey dataset defined by a Croissant schema.
- How to reference and extract data via entity `@id`s, as recommended for robust FAIR data workflows.
- Essential exploratory analysis of mental health scores (e.g., GAD-7, PHQ-9) and subgroup comparison by demographics.

**For more advanced usage, consult the [mlcroissant documentation](https://mlcommons.github.io/croissant/spec/) and the data package's full schema for custom queries and advanced features.**